# CellRanger → PerturbFlow: real lab path

Most CRISPR-screen labs land at a CellRanger 7.x output directory:

```
outs/
├── filtered_feature_bc_matrix.h5         # 10x H5 with GEX + CRISPR features
├── filtered_feature_bc_matrix/
│   ├── features.tsv.gz                   # feature_id, name, type, target gene
│   ├── barcodes.tsv.gz
│   └── matrix.mtx.gz
└── crispr_analysis/
    └── protospacer_calls_per_cell.csv    # cell -> guide assignments
```

This notebook turns that directory into a PerturbFlow run with zero hand-rolled CSVs.

In [ ]:
from pathlib import Path
import perturbflow as pf

CELLRANGER_OUT = Path('data/my_screen/outs')

# 1. Load the gene-expression matrix (gex_only=True drops CRISPR features
#    from X — they're recovered separately from the protospacer CSV).
adata = pf.read_10x_h5(CELLRANGER_OUT / 'filtered_feature_bc_matrix.h5')

# 2. Read the per-cell guide assignment table.
guide_calls = pf.read_cellranger_protospacer_calls(
    CELLRANGER_OUT / 'crispr_analysis' / 'protospacer_calls_per_cell.csv'
)

# 3. Build the guide metadata table from the features file.
guide_metadata = pf.guide_metadata_from_cellranger_features(
    CELLRANGER_OUT / 'filtered_feature_bc_matrix' / 'features.tsv.gz',
    nontargeting_pattern='non-targeting',  # adjust to match your library labels
)

print(f'Cells: {adata.n_obs}, genes: {adata.n_vars}')
print(f'Guide calls: {len(guide_calls)} rows for {guide_calls["cell_barcode"].nunique()} cells')
print(f'Library: {len(guide_metadata)} guides, {guide_metadata["is_control"].sum()} NT')

From here, the rest of the pipeline is identical to any other run:

```python
adata = pf.assign_guides(adata, guide_calls, guide_metadata)
adata = pf.run_mixscape(adata)
de = pf.run_pseudobulk_de(adata)
...
```

Or save the three inputs to disk and use the config-driven runner:

```bash
perturbflow run --config workflow/config.yaml
```

## Common gotchas

**Multi-feature cells.** CellRanger emits one row per cell with
pipe-separated assignments for cells that received multiple guides
(``GeneA_g1|GeneA_g2``). ``read_cellranger_protospacer_calls`` expands
these to one row per (cell, guide) by default; ``assign_guides`` will
then flag them as ``multi-guide``. Pass ``drop_multifeature=True`` if
you want to filter them before assignment instead.

**Non-targeting label.** The default ``nontargeting_pattern`` matches
``'non-targeting'`` (case-insensitive substring) in the
``target_gene_name`` column. Common alternatives:

- Brunello / TKO libraries: ``nontargeting_pattern='Non-Targeting'``
- Many CRISPRi libraries: ``nontargeting_pattern='nt'`` (use with care —
  matches genes starting with ``Nt``)
- Replogle: ``nontargeting_pattern='non-targeting'``

Inspect your features.tsv first if you're unsure.

**Barcode suffixes.** CellRanger appends ``-1`` to barcodes; the
protospacer CSV uses the same. If you aggregated multiple samples with
``cellranger aggr``, the suffix becomes ``-<sample_idx>`` and you need
to make sure the matrix barcodes and the protospacer barcodes agree.